# Notebook 02 — Experiment 1 latency analysis

Canonical analysis entry point for **Experiment 1: MCP-overhead comparison across Cells A / B / C**.

Until all three cells have committed captures, this notebook runs in **preflight mode**: it checks the expected artifact layout (per `docs/runbook.md` §3.4 and `#25`) and writes an availability snapshot to `results/metrics/`. When the captures land, the same notebook rolls forward into the real latency analysis without a structural rewrite.

**Artifact layout expected** (from `scripts/run_experiment.sh`):

```
benchmarks/cell_<A|B|C>_<name>/
├── config.json       # canonical reproducibility config + wandb_run_url
├── summary.json      # aggregate stats: latency_seconds_{mean,p50,p95}, mcp_latency_*, tool_*, etc.
└── raw/<run-id>/
    ├── meta.json             # run metadata + profiling linkage (#27)
    ├── latencies.jsonl       # one row per scenario×trial
    ├── harness.log
    ├── vllm.log              # if LAUNCH_VLLM=1
    └── <date>_<cell>_<model>_<orch>_<mcp>_<scenario>_run<n>.json
```

Run IDs are `<slurm_job_id>_<EXPERIMENT_NAME>` (Slurm) or `local-<YYYYMMDD-HHMMSS>_<EXPERIMENT_NAME>` (local).

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "benchmarks").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook path")

REPO_ROOT = find_repo_root(Path.cwd())
BENCHMARKS_DIR = REPO_ROOT / "benchmarks"
RESULTS_METRICS_DIR = REPO_ROOT / "results" / "metrics"
RESULTS_FIGURES_DIR = REPO_ROOT / "results" / "figures"
RESULTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:   {REPO_ROOT}")
print(f"Python:      {platform.python_version()}")
print(f"pandas:      {pd.__version__}")
print(f"numpy:       {np.__version__}")
print(f"matplotlib:  {matplotlib.__version__}")

## Cells scanned

Notebook 02 is scoped to the three Experiment 1 conditions. Cells Y and Z belong to Experiment 2 and live in Notebook 03.

| Cell | Label | Directory |
|---|---|---|
| A | Direct | `benchmarks/cell_A_direct/` |
| B | MCP baseline | `benchmarks/cell_B_mcp_baseline/` |
| C | MCP optimized | `benchmarks/cell_C_mcp_optimized/` |

Per-cell config lives at `configs/aat_{direct,mcp_baseline,mcp_optimized}.env` (per `docs/experiment1_capture_plan.md` from `91cb21e`). The older `configs/experiment1/exp1_cell_*.env` duplicates exist from the pre-rebase scaffold and should be reconciled before first live run — the notebook keys off `benchmarks/cell_<X>_*/` and is agnostic to which config file was used.

In [ ]:
CELL_SPECS = [
    {"cell": "A", "label": "Direct",        "path": BENCHMARKS_DIR / "cell_A_direct"},
    {"cell": "B", "label": "MCP baseline",  "path": BENCHMARKS_DIR / "cell_B_mcp_baseline"},
    {"cell": "C", "label": "MCP optimized", "path": BENCHMARKS_DIR / "cell_C_mcp_optimized"},
]

SUMMARY_FIELDS = [
    # orchestration / identity
    "experiment_family", "experiment_cell", "orchestration_mode", "mcp_mode",
    "scenario_set_name", "scenario_set_hash", "model_id", "model_provider",
    "serving_stack", "quantization_mode",
    # outcome
    "run_status", "scenarios_attempted", "scenarios_completed",
    "success_rate", "failure_count",
    # latency
    "wall_clock_seconds_total",
    "latency_seconds_mean", "latency_seconds_p50", "latency_seconds_p95",
    # tool / MCP
    "tool_call_count_total", "tool_call_count_mean", "tool_error_count",
    "mcp_latency_seconds_mean", "mcp_latency_seconds_p95",
    "tool_latency_seconds_mean",
    # wandb / judge (may be null in Experiment 1)
    "wandb_run_url",
    "judge_score_mean", "judge_score_p50", "judge_score_p95",
    "judge_pass_rate",
]

META_PROFILING_FIELDS = ["profiling_dir", "profiling_artifact", "profiling_summary"]

def _load_json(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None

def _count_jsonl_rows(path: Path) -> int:
    if path is None or not path.exists():
        return 0
    with path.open() as handle:
        return sum(1 for line in handle if line.strip())

def scan_cell(cell_spec: dict) -> dict:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    run_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()]) if raw_dir.exists() else []
    latest_run = run_dirs[-1] if run_dirs else None

    summary = _load_json(cell_dir / "summary.json")
    config = _load_json(cell_dir / "config.json")
    latencies_path = latest_run / "latencies.jsonl" if latest_run else None
    meta = _load_json(latest_run / "meta.json") if latest_run else None

    row = {
        "cell": cell_spec["cell"],
        "label": cell_spec["label"],
        "cell_dir": str(cell_dir.relative_to(REPO_ROOT)),
        "config_present": (cell_dir / "config.json").exists(),
        "summary_present": (cell_dir / "summary.json").exists(),
        "raw_run_count": len(run_dirs),
        "latest_run_id": latest_run.name if latest_run else "",
        "latency_rows": _count_jsonl_rows(latencies_path) if latencies_path else 0,
    }
    for field in SUMMARY_FIELDS:
        row[field] = summary.get(field) if summary else None
    # Profiling linkage (#27) stamps these onto meta.json for the linked run.
    for field in META_PROFILING_FIELDS:
        row[field] = meta.get(field) if meta else None
    return row

availability_df = pd.DataFrame(scan_cell(spec) for spec in CELL_SPECS)
availability_df

In [ ]:
# Write the cheap availability snapshot so downstream consumers don't have
# to rerun the scan just to see what exists.
availability_path = RESULTS_METRICS_DIR / "notebook02_cell_availability.csv"
availability_df.to_csv(availability_path, index=False)
print(f"Wrote {availability_path.relative_to(REPO_ROOT)}")

display_cols = [
    "cell", "label", "config_present", "summary_present", "raw_run_count",
    "latest_run_id", "latency_rows", "run_status",
    "orchestration_mode", "mcp_mode",
    "latency_seconds_mean", "latency_seconds_p95",
    "mcp_latency_seconds_mean", "tool_error_count",
    "profiling_dir", "wandb_run_url",
]
display(availability_df[display_cols])

## Readiness gate

Notebook 02 only claims Experiment 1 findings once **all three cells** have:

- a committed `config.json`
- a committed `summary.json`
- at least one raw run directory with ≥1 `latencies.jsonl` row

Until then, the notebook stays in preflight mode and only exports the availability snapshot.

In [ ]:
availability_df["analysis_ready"] = (
    availability_df["config_present"]
    & availability_df["summary_present"]
    & (availability_df["raw_run_count"] > 0)
    & (availability_df["latency_rows"] > 0)
)

ready_cells = availability_df.loc[availability_df["analysis_ready"], "cell"].tolist()
missing_cells = availability_df.loc[~availability_df["analysis_ready"], "cell"].tolist()

print(f"Ready cells:   {ready_cells if ready_cells else 'none'}")
print(f"Missing cells: {missing_cells if missing_cells else 'none'}")

analysis_ready = len(missing_cells) == 0
if not analysis_ready:
    print("\nScaffold mode: waiting on real captures for Cells A, B, C.")
    print("Expected captures per #25 come from scripts/run_experiment.sh runs against configs/aat_*.env.")

## Per-scenario latencies

When all three cells are ready, pull `latencies.jsonl` from the **latest** run in each cell, join to a single dataframe, and produce both summary stats and per-scenario plots. Non-fatal degradation: if a cell is missing, the cell is skipped rather than blocking the rest.

In [ ]:
def load_cell_latencies(cell_spec: dict) -> pd.DataFrame:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    run_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
    if not run_dirs:
        return pd.DataFrame()
    latest_run = run_dirs[-1]
    rows = []
    with (latest_run / "latencies.jsonl").open() as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            row["cell"] = cell_spec["cell"]
            row["label"] = cell_spec["label"]
            row["run_id"] = latest_run.name
            rows.append(row)
    return pd.DataFrame(rows)

if analysis_ready:
    latency_df = pd.concat(
        [load_cell_latencies(spec) for spec in CELL_SPECS],
        ignore_index=True,
    )

    latency_summary = (
        latency_df.groupby(["cell", "label"], as_index=False)["latency_seconds"]
        .agg(
            count="count",
            latency_mean="mean",
            latency_p50="median",
            latency_p95=lambda s: s.quantile(0.95),
            latency_max="max",
        )
    )

    summary_path = RESULTS_METRICS_DIR / "notebook02_latency_summary.csv"
    latency_summary.to_csv(summary_path, index=False)
    print(f"Wrote {summary_path.relative_to(REPO_ROOT)}")
    display(latency_summary)
else:
    latency_df = pd.DataFrame()
    latency_summary = pd.DataFrame()
    print("Skipping per-scenario aggregation — cells missing captures.")

## MCP overhead decomposition

Once all three cells are captured, the benchmark claim the paper needs is a clean overhead decomposition:

- **MCP transport overhead** = median(Cell B) − median(Cell A). This is the cost of the JSON-RPC layer with nothing else changed.
- **Optimization delta** = median(Cell B) − median(Cell C). This is what the MCP optimizations (batched tool calls, caching, etc.) actually save versus the naive baseline.
- **Net overhead after optimization** = median(Cell C) − median(Cell A). Ideally this is small; if it's large, MCP still has a floor that optimization can't erase.

All three are computed from per-scenario latencies to avoid Simpson's paradox (summary-level numbers from `summary.json` are an average of averages).

Report at both median (p50) and p95 — the tail is where MCP overhead tends to bite.

In [ ]:
if analysis_ready and not latency_df.empty:
    per_cell = latency_df.groupby("cell")["latency_seconds"]
    p50 = per_cell.median()
    p95 = per_cell.quantile(0.95)

    overhead = pd.DataFrame({
        "metric": [
            "MCP transport overhead (B − A)",
            "Optimization delta (B − C)",
            "Net overhead after optimization (C − A)",
        ],
        "median_seconds": [
            p50.get("B", np.nan) - p50.get("A", np.nan),
            p50.get("B", np.nan) - p50.get("C", np.nan),
            p50.get("C", np.nan) - p50.get("A", np.nan),
        ],
        "p95_seconds": [
            p95.get("B", np.nan) - p95.get("A", np.nan),
            p95.get("B", np.nan) - p95.get("C", np.nan),
            p95.get("C", np.nan) - p95.get("A", np.nan),
        ],
    })

    overhead_path = RESULTS_METRICS_DIR / "notebook02_mcp_overhead.csv"
    overhead.to_csv(overhead_path, index=False)
    print(f"Wrote {overhead_path.relative_to(REPO_ROOT)}")
    display(overhead)
else:
    print("MCP overhead decomposition pending readiness.")

## Figure — latency by cell

Paper-facing plot. p50 bars with p95 error marks, saved to `results/figures/`.

In [ ]:
if analysis_ready and not latency_df.empty:
    summary = latency_summary.set_index("cell").loc[["A", "B", "C"]]
    labels = summary["label"].tolist()
    p50 = summary["latency_p50"].values
    p95 = summary["latency_p95"].values

    fig, ax = plt.subplots(figsize=(7, 4.5))
    colors = ["#4c78a8", "#f58518", "#54a24b"]
    bars = ax.bar(labels, p50, color=colors, edgecolor="black", linewidth=0.5)
    # Overlay p95 as a capped line per bar.
    for x, mid, high in zip(bars, p50, p95):
        cx = x.get_x() + x.get_width() / 2
        ax.plot([cx, cx], [mid, high], color="black", linewidth=1)
        ax.plot([cx - 0.12, cx + 0.12], [high, high], color="black", linewidth=1)

    ax.set_title("Experiment 1 — latency by cell (p50 bar, p95 cap)")
    ax.set_ylabel("End-to-end latency (seconds)")
    ax.set_xlabel("Condition")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()

    figure_path = RESULTS_FIGURES_DIR / "notebook02_latency_comparison.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {figure_path.relative_to(REPO_ROOT)}")
else:
    print("Figure deferred — need captures for all three cells first.")

## Profiling linkage — visibility

`#27` (`01043c5`) wires `capture_around.sh` → `log_profiling_to_wandb.py` → benchmark WandB run, and stamps `profiling_dir` / `profiling_artifact` / `profiling_summary` back into `meta.json`. Surface those fields here so a reviewer can tell at a glance which cells have profiling attached — and which need a rerun with profiling wired in.

In [ ]:
prof_cols = ["cell", "label", "latest_run_id", "profiling_dir", "profiling_artifact", "profiling_summary", "wandb_run_url"]
prof_view = availability_df[prof_cols].copy()
# Truncate URLs + long summary dicts for display only.
prof_view["profiling_summary"] = prof_view["profiling_summary"].apply(
    lambda v: "set" if isinstance(v, dict) and v else ("missing" if v is None else "empty")
)
display(prof_view)

## What changes later

When `#25` captures land, this notebook should not need a new shape. Expected forward deltas:

- split end-to-end latency from tool-only latency when `tool_latency_seconds` is populated per-row
- pull `mcp_latency_seconds` from the per-scenario JSONs when the runner surfaces per-step MCP timings
- join profiling-summary stats (GPU util, memory, power) into the comparison when profiling runs are attached
- export the paper-facing latency figure under `results/figures/notebook02_latency_comparison.png`
- keep `results/metrics/notebook02_cell_availability.csv` as the cheap preflight artifact so CI / smoke checks can read it without spinning up the notebook

When Experiment 2 captures land but not Experiment 1, **do not** promote Experiment 2 metrics into this notebook — they belong to Notebook 03.